# Train / Validation / Test 数据泄露审计

**问题**：`Dataset/Processed/*.csv` 和 `Dataset/Processed_plain/*.csv` 这些数据集，按 `split` 列切成 train / validation / test 三段，是否存在跨 split 的重复数据（泄露）？

## 数据结构
每行有两个核心文本字段：
- **input**：网络规格 + Existing Open Lines + 该状态下的 NodeVoltages / System Loss / System Load
- **output**：最优的 Open Lines + 优化后的 NodeVoltages / System Loss

## 我们要拆开看的原子字段

| 字段 (field) | 含义 | 是"题目"还是"答案" |
|---|---|---|
| `full_row` | input + output 整体拼接 | 行级完全重复 |
| `input_str` | input 完整字符串 | 题目 (含所有题面信息) |
| `output_str` | output 完整字符串 | 答案 (含数值结果) |
| `system_load` | input 里 `System Load=[...]` 数组 | **题目核心**：物理负荷向量，每个 scenario 独有 |
| `existing_open` | input 里 `Open Lines=[...]` | 题目副信息：初始开断线配置 |
| `target_open` | output 里 `Open Lines=[...]` | **答案核心**：最优开断线配置（即 label） |
| `scenario_id` | `(system_load, existing_open)` 联合 | 一次完整 scenario 的唯一标识 |
| `config_to_label` | `(existing_open, target_open)` 联合 | 初始配置→最优配置的映射对 |

## 什么叫泄露 / 重叠（明确口径）

记 A、B 为两个 split（如 train、test）。对某个字段 X，定义：

- `nunique_A` = A 中 X 的不同取值个数
- `nunique_B` = B 中 X 的不同取值个数
- `|A ∩ B|` = A 和 B 在字段 X 上**共有的不同取值个数**（取值集合的交集大小）
- `pct_of_A` = `|A ∩ B| / nunique_A` × 100% → "A 中有多大比例的取值，B 里也出现了"
- `pct_of_B` = `|A ∩ B| / nunique_B` × 100% → "B 中有多大比例的取值，A 里也出现了"
- `jaccard` = `|A ∩ B| / |A ∪ B|` × 100% → 集合相似度（对称）

**例子**：如果 A=train (87 个不同 target_open)、B=test (85 个不同)、`|A∩B|`=76，那么：
- pct_of_A = 76/87 = 87.4% → train 见过的 label 里有 87.4% 在 test 也出现
- pct_of_B = 76/85 = 89.4% → test 出现的 label 里有 89.4% 模型在 train 就见过
- 这两个数都报，方向更清楚。

## 什么是"好"，什么是"差"（判定标准）

重点要区分**题目**和**答案**两类字段：

| 字段类型 | 跨 split 重叠为 0 → | 跨 split 重叠很高 → |
|---|---|---|
| **题目侧**（system_load / input_str / scenario_id / full_row） | ✅ 好：split 是 disjoint 的，没泄露 | ❌ 差：同一道题出现在两个 split，相当于把答案抄给模型 |
| **答案侧**（target_open / output_str / existing_open / config_to_label） | 中性 — label 空间大或采样多样 | 中性 — label 空间小，不是泄露，但说明任务接近"分类到固定几个桶" |

**真正的泄露判定只看题目侧**。答案侧的高重叠反映的是 label 空间大小、不是泄露；但它也是一个有用信号，能告诉我们任务本质上是"在 N 个固定拓扑里挑"还是"组合搜索"。

In [8]:
from pathlib import Path
import re
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

DATASET_ROOT = Path('/Users/town/Codes/RL4DistReconfig/Dataset')

PROCESSED_FILES = sorted((DATASET_ROOT / 'Processed').glob('train_*_nodes.csv'))
PROCESSED_PLAIN_FILES = sorted((DATASET_ROOT / 'Processed_plain').glob('train_*_nodes_plain.csv'))

print('Processed 目录下的数据集：')
for p in PROCESSED_FILES:
    print(' ', p.name)
print('\nProcessed_plain 目录下的数据集：')
for p in PROCESSED_PLAIN_FILES:
    print(' ', p.name)

Processed 目录下的数据集：
  train_136_nodes.csv
  train_33_69_84_nodes.csv
  train_33_nodes.csv
  train_37_nodes.csv
  train_69_nodes.csv
  train_84_nodes.csv

Processed_plain 目录下的数据集：
  train_136_nodes_plain.csv
  train_33_69_84_nodes_plain.csv
  train_33_nodes_plain.csv
  train_37_nodes_plain.csv
  train_69_nodes_plain.csv
  train_84_nodes_plain.csv


## 字段抽取函数

用正则从 `input` / `output` 字符串里抠出每个原子字段。所有比较都基于**字符串原样比较**（不做归一化、不解析成 list），因为 CSV 里的数值精度是固定的，字符串相等即语义相等。

In [9]:
# input 里 `System Load=[...]` —— 该行的物理负荷向量（每个 scenario 唯一）
_LOAD_RE = re.compile(r'System Load=\[(.+?)\]', re.S)
# input/output 里 `Open Lines=[...]` —— 在 input 中是 existing 配置，
# 在 output 中是 target（最优）配置。两个字符串各只含一处。
_OPEN_RE = re.compile(r'Open Lines=\[(.+?)\]', re.S)


def _first(pattern: re.Pattern, text: str) -> str:
    m = pattern.search(text)
    return m.group(1) if m else ''


def extract_fields(df: pd.DataFrame) -> pd.DataFrame:
    """为一份 CSV 一次性抽取所有要比较的字段，返回一个新 DataFrame。"""
    out = pd.DataFrame({'split': df['split']})
    out['full_row'] = df['input'].astype(str) + ' ||| ' + df['output'].astype(str)
    out['input_str'] = df['input'].astype(str)
    out['output_str'] = df['output'].astype(str)
    out['system_load'] = df['input'].map(lambda s: _first(_LOAD_RE, s))
    out['existing_open'] = df['input'].map(lambda s: _first(_OPEN_RE, s))
    out['target_open'] = df['output'].map(lambda s: _first(_OPEN_RE, s))
    out['scenario_id'] = out['system_load'] + '||' + out['existing_open']
    out['config_to_label'] = out['existing_open'] + '->' + out['target_open']
    return out


# 字段的中文说明 + 类型（题目侧 / 答案侧），后面汇总要用
FIELD_META = [
    ('full_row',         '整行 (input + output)',                  '题目+答案'),
    ('input_str',        'input 完整字符串',                        '题目'),
    ('output_str',       'output 完整字符串',                       '答案'),
    ('system_load',      'System Load 负荷向量',                    '题目核心'),
    ('existing_open',    'Existing Open Lines (初始开断线)',         '题目副信息'),
    ('target_open',      'Target Open Lines (最优开断线 = label)',   '答案核心'),
    ('scenario_id',      '(System Load, Existing Open Lines) 联合',  '题目'),
    ('config_to_label',  '(Existing Open, Target Open) 联合',        '配置→标签对'),
]
FIELD_NAMES = [f for f, _, _ in FIELD_META]

## 第一步：每个 split 有多少行，每个字段在每个 split 里有多少不同取值

**这一表回答**："unique 是哪个 split 上的 unique？" —— 我们对每个 split 单独算 nunique。

In [10]:
SPLITS = ['train', 'validation', 'test']


def split_sizes_table(fields: pd.DataFrame) -> pd.DataFrame:
    """输出：每个 split 的行数 + 每个字段在该 split 上的 nunique。"""
    rows = []
    for sp in SPLITS:
        sub = fields[fields['split'] == sp]
        rec = {'split': sp, 'n_rows': len(sub)}
        for f in FIELD_NAMES:
            rec[f'nunique_{f}'] = sub[f].nunique()
        rows.append(rec)
    return pd.DataFrame(rows)

## 第二步：两两 split 之间的重叠（三对都列出来）

**这一表回答**："谁和谁重叠？" —— 显式枚举 `train↔validation`、`train↔test`、`validation↔test` 三对，**每一对都单独一行**。

每一行列出：
- `A`、`B`：哪两个 split 对比
- `field`：在哪个字段上对比
- `nunique_A`、`nunique_B`：两边各自的不同取值数
- `intersection`：交集里有多少个不同取值（`|A ∩ B|`）
- `pct_of_A`：交集占 A 的比例（A 中有多少比例在 B 也出现）
- `pct_of_B`：交集占 B 的比例（B 中有多少比例在 A 也出现）
- `jaccard`：`|A ∩ B| / |A ∪ B|`（对称指标）

In [11]:
PAIRS = [('train', 'validation'), ('train', 'test'), ('validation', 'test')]


def pairwise_overlap_table(fields: pd.DataFrame) -> pd.DataFrame:
    """对每个字段、每对 split，列出详细的重叠指标。"""
    # 预先把每个 split 在每个字段上的取值集合算出来
    sets = {f: {sp: set(fields.loc[fields['split'] == sp, f]) for sp in SPLITS}
            for f in FIELD_NAMES}
    rows = []
    for f in FIELD_NAMES:
        for a, b in PAIRS:
            sa, sb = sets[f][a], sets[f][b]
            inter = len(sa & sb)
            union = len(sa | sb)
            rows.append({
                'field': f,
                'A': a,
                'B': b,
                f'nunique_A': len(sa),
                f'nunique_B': len(sb),
                'intersection': inter,
                'pct_of_A': round(inter / len(sa) * 100, 2) if sa else float('nan'),
                'pct_of_B': round(inter / len(sb) * 100, 2) if sb else float('nan'),
                'jaccard': round(inter / union * 100, 2) if union else float('nan'),
            })
    return pd.DataFrame(rows)

## 第三步：三个 split **同时**都出现的取值

**这一表回答**："跨 split 究竟是三个都跨呢？还是某两个跨？"

前面两两对比看的是 "至少在哪两个里都出现"；这里看 "三个 split 同时都包含的取值数 `|train ∩ val ∩ test|`"，以及 "恰好只在两个里、第三个里没有" 的数量。

In [12]:
def three_way_overlap_table(fields: pd.DataFrame) -> pd.DataFrame:
    """对每个字段，列出：
       - 三个 split 都有 (|train ∩ val ∩ test|)
       - 恰好两个 split 有 (三种)
       - 只有一个 split 有 (三种)
       这些数加起来等于该字段在整份数据上的总不同取值数。"""
    rows = []
    for f in FIELD_NAMES:
        S = {sp: set(fields.loc[fields['split'] == sp, f]) for sp in SPLITS}
        T, V, Te = S['train'], S['validation'], S['test']
        all_three = T & V & Te
        only_tv = (T & V) - Te
        only_tt = (T & Te) - V
        only_vt = (V & Te) - T
        only_t = T - V - Te
        only_v = V - T - Te
        only_te = Te - T - V
        total_unique = len(T | V | Te)
        rows.append({
            'field': f,
            'in_all_3': len(all_three),
            'only_train_val': len(only_tv),
            'only_train_test': len(only_tt),
            'only_val_test': len(only_vt),
            'only_train': len(only_t),
            'only_val': len(only_v),
            'only_test': len(only_te),
            'total_distinct': total_unique,
        })
    return pd.DataFrame(rows)

## 第四步：单数据集判定（题目侧 vs 答案侧）

**这一段回答**："什么算差，什么算好"。我们对每个数据集明确给出结论，分两类字段：

- **题目侧字段**（`input_str`, `system_load`, `scenario_id`, `full_row`）任何跨 split 重叠 > 0 → 🚨 真正的泄露
- **答案侧字段**（`target_open`, `output_str`, `existing_open`, `config_to_label`）跨 split 重叠不算泄露，但报出来告诉你 label 空间多大，模型是在做"组合构造"还是"分类到 N 个桶"。

In [13]:
# 题目侧字段集合（任何跨 split 重叠都算泄露）
QUESTION_FIELDS = {'full_row', 'input_str', 'system_load', 'scenario_id'}
# 答案侧字段集合（重叠不是泄露，但报来看 label 空间）
ANSWER_FIELDS = {'output_str', 'existing_open', 'target_open', 'config_to_label'}


def per_dataset_verdict(overlap_df: pd.DataFrame, sizes_df: pd.DataFrame) -> str:
    """输出一段人类可读的结论。"""
    lines = []
    # 1) 题目侧：有没有任何跨 split 重叠？
    question_leaks = overlap_df[
        overlap_df['field'].isin(QUESTION_FIELDS) & (overlap_df['intersection'] > 0)
    ]
    if question_leaks.empty:
        lines.append('✅ 题目侧 (input / System Load / scenario_id / full_row)：'
                     '所有三对 split 之间的重叠都是 0 个，没有真正的数据泄露。')
    else:
        lines.append('🚨 题目侧检测到重叠（这就是数据泄露）：')
        for _, r in question_leaks.iterrows():
            lines.append(f"  - 字段 [{r['field']}] 在 {r['A']} ∩ {r['B']} 上有 "
                         f"{r['intersection']} 个共有取值 "
                         f"(占 {r['A']} 的 {r['pct_of_A']}%，占 {r['B']} 的 {r['pct_of_B']}%)")

    # 2) 答案侧：label 空间大小 + 重叠情况
    lines.append('')
    lines.append('📊 答案侧 (target_open = label) 的情况：')
    tgt = overlap_df[overlap_df['field'] == 'target_open']
    # 取 train/val/test 各自的 nunique_target_open
    nun = sizes_df.set_index('split')['nunique_target_open'].to_dict()
    lines.append(f"  - 不同 Target Open Lines 数量：train={nun.get('train')}, "
                 f"validation={nun.get('validation')}, test={nun.get('test')}")
    for _, r in tgt.iterrows():
        lines.append(f"  - {r['A']} ∩ {r['B']}：交集 {r['intersection']} 个标签，"
                     f"占 {r['A']} 的 {r['pct_of_A']}%，占 {r['B']} 的 {r['pct_of_B']}%，"
                     f"Jaccard={r['jaccard']}%")
    # 经验阈值：>50% 当作"label 空间太小，任务接近分类"提醒
    big_overlap = tgt[tgt['pct_of_B'] >= 50]
    if not big_overlap.empty:
        lines.append('  ⚠️ Target Open Lines 跨 split 重叠率 ≥50%：label 空间很小，'
                     '模型实际任务接近"从固定的几个拓扑里挑一个"，不是组合搜索。')
    return '\n'.join(lines)

## 第五步：跑通所有 CSV，每个数据集都打印

对每个 CSV 依次输出：
1. 每个 split 的行数 + 每个字段的 nunique
2. 三对 split × 八个字段 的两两重叠详表
3. 三 split 同时有 / 只两 split 有 / 只一 split 有 的细分表
4. 人类可读判定

In [14]:
ALL_FILES = list(PROCESSED_FILES) + list(PROCESSED_PLAIN_FILES)
results = {}  # 留给后面汇总用

for csv_path in ALL_FILES:
    print('\n' + '=' * 90)
    print(f'数据集：{csv_path.relative_to(DATASET_ROOT)}')
    print('=' * 90)
    df = pd.read_csv(csv_path)
    if 'split' not in df.columns:
        print('  ⚠️ 没有 split 列，跳过')
        continue

    fields = extract_fields(df)

    # 1) 每个 split 的规模
    sizes = split_sizes_table(fields)
    print('\n[1] 每个 split 的行数 + 每个字段的不同取值数 (nunique)：')
    print(sizes.to_string(index=False))

    # 2) 两两重叠详表
    overlap = pairwise_overlap_table(fields)
    print('\n[2] 两两 split 之间的重叠（每行 = 一对 split × 一个字段）：')
    print(overlap.to_string(index=False))

    # 3) 三向重叠
    triple = three_way_overlap_table(fields)
    print('\n[3] 三 split 同时有 / 只有两个 split 有 / 只有一个 split 有：')
    print(triple.to_string(index=False))

    # 4) 判定
    print('\n[4] 结论：')
    print(per_dataset_verdict(overlap, sizes))

    results[csv_path.name] = {'sizes': sizes, 'overlap': overlap, 'triple': triple}


数据集：Processed/train_136_nodes.csv

[1] 每个 split 的行数 + 每个字段的不同取值数 (nunique)：
     split  n_rows  nunique_full_row  nunique_input_str  nunique_output_str  nunique_system_load  nunique_existing_open  nunique_target_open  nunique_scenario_id  nunique_config_to_label
     train    1666              1666               1666                1666                 1666                   1666                 1666                 1666                     1666
validation    1667              1667               1667                1667                 1667                   1667                 1667                 1667                     1667
      test    1666              1666               1666                1666                 1666                   1666                 1666                 1666                     1666

[2] 两两 split 之间的重叠（每行 = 一对 split × 一个字段）：
          field          A          B  nunique_A  nunique_B  intersection  pct_of_A  pct_of_B  jaccard
       full_row      train va

## 第六步：跨数据集的总览表

把所有数据集摆在一起对比关键指标。每行 = 一个 (数据集, 字段)，列出：

- 三个 split 在该字段上的 nunique
- 三对 split 的交集大小（直接给原始数字，不再用"最差"这种模糊词）
- 该字段是"题目"还是"答案"，决定怎么解读

In [15]:
field_kind = {f: kind for f, _, kind in FIELD_META}

summary_rows = []
for name, r in results.items():
    sizes = r['sizes'].set_index('split')
    ov = r['overlap']
    for f in FIELD_NAMES:
        sub = ov[ov['field'] == f].set_index(['A', 'B'])
        summary_rows.append({
            'dataset': name,
            'field': f,
            'kind': field_kind[f],
            'nunique_train': sizes.loc['train', f'nunique_{f}'],
            'nunique_val':   sizes.loc['validation', f'nunique_{f}'],
            'nunique_test':  sizes.loc['test', f'nunique_{f}'],
            '|train∩val|':   sub.loc[('train', 'validation'), 'intersection'],
            '|train∩test|':  sub.loc[('train', 'test'), 'intersection'],
            '|val∩test|':    sub.loc[('validation', 'test'), 'intersection'],
        })
summary = pd.DataFrame(summary_rows)
print('全部数据集 × 全部字段 的关键数字一览：\n')
print(summary.to_string(index=False))

全部数据集 × 全部字段 的关键数字一览：

                       dataset           field   kind  nunique_train  nunique_val  nunique_test  |train∩val|  |train∩test|  |val∩test|
           train_136_nodes.csv        full_row  题目+答案           1666         1667          1666            0             0           0
           train_136_nodes.csv       input_str     题目           1666         1667          1666            0             0           0
           train_136_nodes.csv      output_str     答案           1666         1667          1666            0             0           0
           train_136_nodes.csv     system_load   题目核心           1666         1667          1666            0             0           0
           train_136_nodes.csv   existing_open  题目副信息           1666         1667          1666            0             0           0
           train_136_nodes.csv     target_open   答案核心           1666         1667          1666            0             0           0
           train_136_nodes.csv  

## 第七步：题目侧 vs 答案侧 的关键结论

把上面的总览拆成两张：
1. **题目侧字段**：所有数字都应该是 0，有非零就是真泄露
2. **答案侧字段**：数字大小反映 label 空间，越大越接近"分类任务"

In [16]:
print('=' * 90)
print('题目侧字段（任何跨 split 交集 > 0 都是泄露）：')
print('=' * 90)
q = summary[summary['field'].isin(QUESTION_FIELDS)]
print(q.to_string(index=False))

leaked = q[(q['|train∩val|'] > 0) | (q['|train∩test|'] > 0) | (q['|val∩test|'] > 0)]
print('\n→ 题目侧泄露行数：', len(leaked))
if leaked.empty:
    print('  ✅ 全部数据集，题目侧 train/validation/test 完全 disjoint。没有数据泄露。')
else:
    print('  🚨 以下行存在泄露：')
    print(leaked.to_string(index=False))

print('\n' + '=' * 90)
print('答案侧字段（不是泄露，但反映 label 空间大小 / 任务本质）：')
print('=' * 90)
a = summary[summary['field'].isin(ANSWER_FIELDS)].copy()
# 加一个直观的 pct_of_test = |train∩test| / nunique_test
a['pct_test_label_seen_in_train'] = (a['|train∩test|'] / a['nunique_test'] * 100).round(1)
print(a.to_string(index=False))

题目侧字段（任何跨 split 交集 > 0 都是泄露）：
                       dataset       field  kind  nunique_train  nunique_val  nunique_test  |train∩val|  |train∩test|  |val∩test|
           train_136_nodes.csv    full_row 题目+答案           1666         1667          1666            0             0           0
           train_136_nodes.csv   input_str    题目           1666         1667          1666            0             0           0
           train_136_nodes.csv system_load  题目核心           1666         1667          1666            0             0           0
           train_136_nodes.csv scenario_id    题目           1666         1667          1666            0             0           0
      train_33_69_84_nodes.csv    full_row 题目+答案          17520        17520         17520            0             0           0
      train_33_69_84_nodes.csv   input_str    题目          17520        17520         17520            0             0           0
      train_33_69_84_nodes.csv system_load  题目核心          17